# Three-column state diagram

Minimal notebook containing only the cells needed to load the data and plot:

- `df1`: connectivity,
- `df2`: dominant orientation,
- `df3`: $q_6$ crystallinity from `MAG2P_order_parameters-2026-1-21-10:27:28.pickle`,
- `df4`: $q_4$ crystallinity from `MAG2P_order_parameters-2026-6-17-10:55:21.pickle`.

The third column contains $q_6$ above $q_4$, both with the same color map and numerical scale.


In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import colorsys
import hashlib
import cmasher as cmr

# Required for PDF export:
# %pip install -U kaleido cmasher scipy

LAMBDA_MAX = 20
TICK_FONT = 16
TITLE_FONT = 18


In [2]:
def _find_peaks_fallback(y, min_height=None):
    y = np.asarray(y, dtype=float)
    idx = np.where((y[1:-1] > y[:-2]) & (y[1:-1] >= y[2:]))[0] + 1
    if min_height is not None:
        idx = idx[y[idx] >= min_height]
    return idx


def add_orientation_peaks(
    df: pd.DataFrame,
    shift_col="shift",
    lambda_col="lambda",
    angle_min=0.0,
    angle_max=3.14,
    prominence=0.02,
    min_height=None,
    max_peaks_to_show=8,

    # --- NEW: merge 60° and 120° peaks ---
    merge_60_120=True,
    merge_tol_rad=0.12,   # ~6.9 degrees; tune if needed
):
    """
    Adds:
      - peaks_theta, peaks_height (all peaks after optional merging)
      - dom_theta, dom_height     (highest peak by height)
      - sec_theta, sec_height     (2nd highest peak by height)
      - hover_text
    """
    out = df.copy()

    non_hist = {shift_col, lambda_col}
    hist_cols = [c for c in out.columns if c not in non_hist]

    angles = np.array([float(c) for c in hist_cols], dtype=float)
    order = np.argsort(angles)
    angles = angles[order]
    hist_cols = [hist_cols[i] for i in order]

    H = out[hist_cols].to_numpy(dtype=float)

    try:
        from scipy.signal import find_peaks
        use_scipy = True
    except Exception:
        use_scipy = False

    # canonical merge target: pi/3 (60 degrees)
    th60 = np.pi / 3
    th120 = 2 * np.pi / 3

    def merge_peaks(p_theta, p_height):
        """Merge peaks near 60 and 120 into a single bin at pi/3 by summing heights."""
        if not merge_60_120:
            return p_theta, p_height

        p_theta = np.asarray(p_theta, float)
        p_height = np.asarray(p_height, float)

        mask60 = np.abs(p_theta - th60) <= merge_tol_rad
        mask120 = np.abs(p_theta - th120) <= merge_tol_rad
        mask_merge = mask60 | mask120

        if not np.any(mask_merge):
            return p_theta.tolist(), p_height.tolist()

        merged_h = float(np.sum(p_height[mask_merge]))
        keep_theta = p_theta[~mask_merge].tolist()
        keep_h = p_height[~mask_merge].tolist()

        # add merged peak at canonical 60°
        keep_theta.append(float(th60))
        keep_h.append(merged_h)

        return keep_theta, keep_h

    peaks_theta_all, peaks_height_all = [], []
    dom_theta, dom_height = [], []
    sec_theta, sec_height = [], []
    hover_text = []

    for row_y in H:
        y = np.asarray(row_y, dtype=float)

        if use_scipy:
            peaks, props = find_peaks(y, prominence=prominence, height=min_height)
        else:
            peaks = _find_peaks_fallback(y, min_height=min_height)
            props = {}

        if len(peaks) == 0:
            peaks_theta_all.append([])
            peaks_height_all.append([])
            dom_theta.append(np.nan); dom_height.append(np.nan)
            sec_theta.append(np.nan); sec_height.append(np.nan)
            hover_text.append("No peaks found")
            continue

        p_theta = angles[peaks].tolist()
        p_height = y[peaks].tolist()

        # --- NEW: merge 60/120 group ---
        p_theta, p_height = merge_peaks(p_theta, p_height)

        # rank by height desc
        pairs = sorted(zip(p_theta, p_height), key=lambda t: t[1], reverse=True)

        d_theta, d_height = pairs[0]
        if len(pairs) > 1:
            s_theta, s_height = pairs[1]
        else:
            s_theta, s_height = np.nan, np.nan

        # hover
        peaks_sorted = pairs[:max_peaks_to_show]
        ht_lines = [
            f"dominant θ={d_theta:.3f}, h={d_height:.4g}",
            (f"2nd θ={s_theta:.3f}, h={s_height:.4g}" if np.isfinite(s_theta) else "2nd: (none)"),
            "all peaks:"
        ]
        ht_lines += [f"  θ={t:.3f}, h={h:.4g}" for t, h in peaks_sorted]
        if len(pairs) > max_peaks_to_show:
            ht_lines.append(f"  … (+{len(pairs)-max_peaks_to_show} more)")
        ht = "<br>".join(ht_lines)

        peaks_theta_all.append(list(map(float, [t for t, _ in pairs])))
        peaks_height_all.append(list(map(float, [h for _, h in pairs])))
        dom_theta.append(float(d_theta)); dom_height.append(float(d_height))
        sec_theta.append(float(s_theta) if np.isfinite(s_theta) else np.nan)
        sec_height.append(float(s_height) if np.isfinite(s_height) else np.nan)
        hover_text.append(ht)

    out["peaks_theta"] = peaks_theta_all
    out["peaks_height"] = peaks_height_all
    out["dom_theta"] = dom_theta
    out["dom_height"] = dom_height
    out["sec_theta"] = sec_theta
    out["sec_height"] = sec_height
    out["hover_text"] = hover_text
    return out

In [3]:
import pandas as pd

# ============================================================
# df1: topology / connectivity
# ============================================================
topology_file = (
    "/home/karner/Documents/github/MagneticParticles/rigid_magnetic/results/"
    "MAG2P_order_parameters_per_cluster-2026-1-13-18:15:54.pickle"
)

# Alternative:
# topology_file = (
#     "/home/karner/Documents/github/MagneticParticles/rigid_magnetic/results/"
#     "MAG2P_order_parameters_per_cluster-CUTOFF_1.6-2026-2-13-11:51:35.pickle"
# )

df1 = pd.read_pickle(topology_file).fillna(0)


# ============================================================
# df2: orientation
# ============================================================
order_parameter_file = (
    "/home/karner/Documents/github/MagneticParticles/rigid_magnetic/results/"
    "MAG2P_order_parameters-2026-1-21-10:27:28.pickle"
)

df_orientation = pd.read_pickle(order_parameter_file).fillna(0)

df_orientation = df_orientation.drop(
    columns=[
        "std_bonds_1_8",
        "std_bonds_1_5",
        "std_size",
        "std_radius_of_gyration",
        "std_second_neighbours",
        "mean_Psi_6",
        "mean_Psi_4",
        "mean_q4",
        "mean_q6",
        "p_q6",
        "p_q4",
    ],
    errors="ignore",
)

# Remove columns that contain only zeros.
df_orientation = df_orientation.loc[
    :,
    (df_orientation != 0).any(axis=0),
].copy()

start = 33
end = 62
df_orientation = df_orientation.drop(
    df_orientation.columns[start:end],
    axis=1,
)

dg_orientation = df_orientation[["lambda", "shift"]].copy()
dg_orientation[0] = 0

orientation_columns = df_orientation.columns[9:33]
dg_orientation[orientation_columns] = df_orientation[orientation_columns]

dg_orientation[3.14] = 0

dh = (
    dg_orientation
    .groupby(["shift", "lambda"])
    .sum()
    .reset_index()
)

df_cut = dh.loc[dh["lambda"] <= 20].copy()

print("Orientation columns:")
print(df_cut.columns)

if "add_orientation_peaks" not in globals():
    raise NameError(
        "add_orientation_peaks is not defined. "
        "Run the earlier cell containing its complete definition first."
    )

df2 = add_orientation_peaks(
    df_cut,
    prominence=0.02,
    min_height=0.2,
    merge_60_120=True,
    merge_tol_rad=0.12,
)


# ============================================================
# df3: q6 crystallinity
# ============================================================
df3 = (
    pd.read_pickle(order_parameter_file)
    [["shift", "lambda", "p_q6"]]
    .fillna(0)
)

df3 = df3.loc[df3["lambda"] <= 20].copy()

print("\ndf3 columns:")
print(df3.columns)
display(df3.head())


# ============================================================
# df4: q4 crystallinity
# ============================================================
q4_file = (
    "/home/karner/Documents/github/MagneticParticles/rigid_magnetic/results/"
    "MAG2P_order_parameters-2026-6-17-10:55:21.pickle"
)

df4 = (
    pd.read_pickle(q4_file)
    [["shift", "lambda", "p_q4"]]
    .fillna(0)
)

df4 = df4.loc[df4["lambda"] <= 20].copy()

print("\ndf4 columns:")
print(df4.columns)
display(df4.head())


/tmp/ipykernel_442300/2625313901.py:28: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_orientation = pd.read_pickle(order_parameter_file).fillna(0)


Orientation columns:
Index([            'shift',            'lambda',                   0,
       0.12566370614359174, 0.25132741228718347,  0.3769911184307752,
        0.5026548245743669,  0.6283185307179586,  0.7539822368615504,
        0.8796459430051422,  1.0053096491487339,  1.1309733552923256,
        1.2566370614359172,  1.3823007675795091,  1.5079644737231008,
        1.6336281798666925,  1.7592918860102844,   1.884955592153876,
        2.0106192982974678,  2.1362830044410597,   2.261946710584651,
         2.387610416728243,  2.5132741228718345,  2.6389378290154264,
        2.7646015351590183,  2.8902652413026098,  3.0159289474462017,
                      3.14],
      dtype='object')

df3 columns:
Index(['shift', 'lambda', 'p_q6'], dtype='object')


/tmp/ipykernel_442300/2625313901.py:101: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(0)


,shift,lambda,p_q6
5,0.40,2.3,0.048
6,0.30,1.0,0.000
7,0.50,2.5,0.007
11,0.05,1.5,0.000
12,0.45,10.0,0.036



df4 columns:
Index(['shift', 'lambda', 'p_q4'], dtype='object')


/tmp/ipykernel_442300/2625313901.py:122: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(0)


,shift,lambda,p_q4
5,0.40,2.3,0.039
6,0.30,1.0,0.000
7,0.50,2.5,0.014
11,0.05,1.5,0.000
12,0.45,10.0,0.133


In [19]:
def cmasher_to_plotly_colorscale(cmap, n=256):
    xs = np.linspace(0, 1, n)
    colors = []

    for x in xs:
        r, g, b, a = cmap(x)
        colors.append([
            float(x),
            f"rgba({int(r * 255)},{int(g * 255)},{int(b * 255)},{a:.3f})",
        ])

    return colors


# ============================================================
# 1) Topological / connectivity panel
# ============================================================
STRUCTS = [
    "chain",
    "liquid",
    "complex_network",
    "ring",
    "strongly_clustered",
]

BEND = 0.35

BASE_HEX = {
    "chain": "#000000",
    "liquid": "#86878A",
    "complex_network": "#1CA879",
    "ring": "#591FDF",
    "strongly_clustered": "#EEBF25",
}

STRUCT_LABELS = {
    "chain": "chain",
    "liquid": "liquid",
    "complex_network": "complex network",
    "ring": "ring",
    "strongly_clustered": "strongly clustered",
}


def hex_to_rgb255(hex_color):
    h = hex_color.lstrip("#")[:6]
    return np.array(
        [int(h[i:i + 2], 16) for i in (0, 2, 4)],
        dtype=float,
    )


BASE = {
    name: hex_to_rgb255(color)
    for name, color in BASE_HEX.items()
}


def rgb_str(rgb):
    rgb = np.clip(np.round(rgb), 0, 255).astype(int)
    return f"rgb({rgb[0]},{rgb[1]},{rgb[2]})"


def pair_key(a, b):
    return "|".join(sorted([a, b]))


def pair_mid_color(a, b):
    key = pair_key(a, b).encode("utf-8")

    hue = (
        int(hashlib.md5(key).hexdigest()[:8], 16) % 360
    ) / 360.0

    r, g, b = colorsys.hsv_to_rgb(
        hue,
        0.65,
        0.97,
    )

    return np.array([r, g, b]) * 255.0


def bent_mix(Ca, Cb, midpoint, t, bend=BEND):
    straight_midpoint = 0.5 * (Ca + Cb)
    control_point = (
        (1 - bend) * straight_midpoint
        + bend * midpoint
    )

    return (
        (1 - t) ** 2 * Ca
        + 2 * (1 - t) * t * control_point
        + t ** 2 * Cb
    )


def make_topological_traces(
    df1,
    shift_col="shift",
    lambda_col="lambda",
):
    df_filt = df1.loc[
        df1[lambda_col] <= LAMBDA_MAX
    ].copy()

    counts = (
        df_filt
        .groupby(
            [lambda_col, shift_col, "structure_type"]
        )["cluster_size"]
        .sum()
        .unstack("structure_type", fill_value=0)
    )

    for structure in STRUCTS:
        if structure not in counts.columns:
            counts[structure] = 0

    counts = counts[STRUCTS]

    totals = counts.sum(axis=1)
    percentages = counts.div(
        totals,
        axis=0,
    ).fillna(0.0)

    P = percentages.to_numpy()

    top2_idx = np.argsort(-P, axis=1)[:, :2]
    top2_values = np.take_along_axis(
        P,
        top2_idx,
        axis=1,
    )

    structure_array = np.array(
        STRUCTS,
        dtype=object,
    )

    s1 = structure_array[top2_idx[:, 0]]
    s2 = structure_array[top2_idx[:, 1]]

    p1 = top2_values[:, 0]
    p2 = top2_values[:, 1]

    top2_mass = np.maximum(
        p1 + p2,
        1e-12,
    )

    top2 = pd.DataFrame({
        lambda_col: percentages.index.get_level_values(
            lambda_col
        ),
        shift_col: percentages.index.get_level_values(
            shift_col
        ),
        "s1": s1,
        "s2": s2,
        "p1": p1,
        "p2": p2,
        "top2_mass": top2_mass,
        "total_cluster_size": totals.to_numpy(),
    }).reset_index(drop=True)

    lambda_values = np.sort(
        top2[lambda_col].unique()
    )

    shift_values = np.sort(
        top2[shift_col].unique()
    )

    height = len(lambda_values)
    width = len(shift_values)

    def pivot_grid(value_column):
        return (
            top2
            .pivot(
                index=lambda_col,
                columns=shift_col,
                values=value_column,
            )
            .reindex(
                index=lambda_values,
                columns=shift_values,
            )
            .to_numpy()
        )

    s1_grid = pivot_grid("s1")
    s2_grid = pivot_grid("s2")
    p1_grid = pivot_grid("p1")
    p2_grid = pivot_grid("p2")
    mass_grid = pivot_grid("top2_mass")
    total_grid = pivot_grid("total_cluster_size")

    t_grid = np.divide(
        p2_grid,
        mass_grid,
        out=np.zeros_like(
            p2_grid,
            dtype=float,
        ),
        where=mass_grid != 0,
    )

    color_grid = np.empty(
        (height, width),
        dtype=object,
    )

    for i in range(height):
        for j in range(width):
            a = s1_grid[i, j]
            b = s2_grid[i, j]

            if not isinstance(a, str) or not isinstance(b, str):
                color_grid[i, j] = "rgb(255,255,255)"
                continue

            rgb = bent_mix(
                BASE[a],
                BASE[b],
                pair_mid_color(a, b),
                float(t_grid[i, j]),
                bend=BEND,
            )

            color_grid[i, j] = rgb_str(rgb)

    n_cells = height * width
    ids = np.arange(n_cells).reshape(
        height,
        width,
    )

    denominator = n_cells - 1 if n_cells > 1 else 1
    z = ids / denominator

    colorscale = []
    flat_colors = color_grid.reshape(-1)

    for k in range(n_cells):
        v0 = k / denominator

        if n_cells > 1 and k < n_cells - 1:
            v1 = (k + 1) / denominator
        else:
            v1 = 1.0

        color = flat_colors[k]
        colorscale.append((v0, color))
        colorscale.append((v1, color))

    customdata = np.stack(
        [
            s1_grid.astype(object),
            s2_grid.astype(object),
            p1_grid,
            p2_grid,
            mass_grid,
            total_grid,
        ],
        axis=-1,
    )

    topology_heatmap = go.Heatmap(
        x=shift_values,
        y=lambda_values,
        z=z,
        zmin=0,
        zmax=1,
        colorscale=colorscale,
        showscale=False,
        customdata=customdata,
        hovertemplate=(
            "shift=%{x}<br>"
            "λ=%{y}<br>"
            "<b>top 2</b>: "
            "%{customdata[0]} (%{customdata[2]:.1%}) vs "
            "%{customdata[1]} (%{customdata[3]:.1%})<br>"
            "top-2 mass=%{customdata[4]:.1%}<br>"
            "total cluster size=%{customdata[5]:.0f}"
            "<extra></extra>"
        ),
        name="topology",
    )

    legend_traces = []

    for structure in STRUCTS:
        legend_traces.append(
            go.Scatter(
                x=[None],
                y=[None],
                mode="markers",
                marker=dict(
                    symbol="square",
                    size=12,
                    color=BASE_HEX[structure],
                ),
                name=STRUCT_LABELS[structure],
                hoverinfo="skip",
                showlegend=True,
            )
        )

    return topology_heatmap, legend_traces


# ============================================================
# 2) Orientation panel
# ============================================================
def make_orientation_trace(
    df2,
    shift_col="shift",
    lambda_col="lambda",
    z_col="dom_theta",
    hover_col="hover_text",
    cmap_name="cmr.swamp",
):
    d = df2.loc[
        df2[lambda_col] <= LAMBDA_MAX
    ].copy()

    if "dom_height" in d.columns:
        d = (
            d.sort_values(
                "dom_height",
                ascending=False,
            )
            .drop_duplicates(
                [lambda_col, shift_col]
            )
        )
    else:
        d = d.drop_duplicates(
            [lambda_col, shift_col]
        )

    shift_values = np.sort(
        d[shift_col].unique()
    )

    lambda_values = np.sort(
        d[lambda_col].unique()
    )

    z = np.full(
        (len(lambda_values), len(shift_values)),
        np.nan,
        dtype=float,
    )

    custom = np.empty(
        (len(lambda_values), len(shift_values)),
        dtype=object,
    )

    idx_x = {
        value: i
        for i, value in enumerate(shift_values)
    }

    idx_y = {
        value: i
        for i, value in enumerate(lambda_values)
    }

    for _, row in d.iterrows():
        i = idx_y[row[lambda_col]]
        j = idx_x[row[shift_col]]

        z[i, j] = float(row[z_col])
        custom[i, j] = row.get(hover_col, "")

    cmap = cmr.get_sub_cmap(
        cmap_name,
        0.05,
        0.95,
    )

    colorscale = cmasher_to_plotly_colorscale(
        cmap,
        256,
    )

    return go.Heatmap(
        x=shift_values,
        y=lambda_values,
        z=z,
        zmin=0,
        zmax=np.pi,
        colorscale=colorscale,
        customdata=custom,
        hovertemplate=(
            "shift=%{x}<br>"
            "λ=%{y}<br>"
            "dominant θ=%{z:.4f}<br>"
            "%{customdata}"
            "<extra></extra>"
        ),
        colorbar=dict(
            tickvals=[
                0,
                np.pi / 4,
                np.pi / 2,
                3 * np.pi / 4,
                np.pi,
            ],
            ticktext=[
                "0",
                "π/4",
                "π/2",
                "3π/4",
                "π",
            ],
            thickness=18,
        ),
        name="orientation",
    )


# ============================================================
# 3) q6 and q4 scalar crystallinity panels
# ============================================================
def make_scalar_mean_trace(
    df3,
    value_col,
    trace_name,
    shift_col="shift",
    lambda_col="lambda",
    cmap_name="cmr.cosmic",
    vmin=0.0,
    vmax=0.4,
    show_colorbar=True,
):
    d = df3.loc[
        df3[lambda_col] <= LAMBDA_MAX
    ].copy()

    d = (
        d.groupby(
            [lambda_col, shift_col],
            as_index=False,
        )[value_col]
        .mean()
    )

    shift_values = np.sort(
        d[shift_col].unique()
    )

    lambda_values = np.sort(
        d[lambda_col].unique()
    )

    z = np.full(
        (len(lambda_values), len(shift_values)),
        np.nan,
        dtype=float,
    )

    idx_x = {
        value: i
        for i, value in enumerate(shift_values)
    }

    idx_y = {
        value: i
        for i, value in enumerate(lambda_values)
    }

    for _, row in d.iterrows():
        z[
            idx_y[row[lambda_col]],
            idx_x[row[shift_col]],
        ] = float(row[value_col])

    cmap = cmr.get_sub_cmap(
        cmap_name,
        0.05,
        0.95,
    )

    colorscale = cmasher_to_plotly_colorscale(
        cmap,
        256,
    )

    return go.Heatmap(
        name=trace_name,
        x=shift_values,
        y=lambda_values,
        z=z,
        zmin=vmin,
        zmax=vmax,
        colorscale=colorscale,
        showscale=show_colorbar,
        hovertemplate=(
            "shift=%{x}<br>"
            "λ=%{y}<br>"
            f"{value_col}=%{{z:.4f}}"
            "<extra></extra>"
        ),
        colorbar=dict(
            title="ordered fraction",
            thickness=18,
        ),
    )


# ============================================================
# Combined layout
# ============================================================
def make_combined_state_diagram(
    df1,
    df2,
    df3,
    df4,
    orientation_cmap="cmr.swamp",
    cryst_cmap="cmr.cosmic",
    cryst_vmin=0.0,
    cryst_vmax=0.4,
    width=1650,
    height=760,
):
    topology_heatmap, topology_legend = (
        make_topological_traces(df1)
    )

    orientation_heatmap = make_orientation_trace(
        df2,
        cmap_name=orientation_cmap,
    )

    # q6 and q4 deliberately receive the same cmap,
    # zmin and zmax so that equal values have equal colors.
    q6_heatmap = make_scalar_mean_trace(
        df3,
        value_col="p_q6",
        trace_name="q6 crystallinity",
        cmap_name=cryst_cmap,
        vmin=cryst_vmin,
        vmax=cryst_vmax,
        show_colorbar=True,
    )

    q4_heatmap = make_scalar_mean_trace(
        df4,
        value_col="p_q4",
        trace_name="q4 crystallinity",
        cmap_name=cryst_cmap,
        vmin=cryst_vmin,
        vmax=cryst_vmax,
        show_colorbar=False,
    )

    fig = make_subplots(
        rows=2,
        cols=3,
        specs=[
            [
                {"rowspan": 2},
                {"rowspan": 2},
                {},
            ],
            [
                None,
                None,
                {},
            ],
        ],
        column_widths=[1.0, 1.0, 1.0],
        row_heights=[0.5, 0.5],
        horizontal_spacing=0.075,
        vertical_spacing=0.13,
        subplot_titles=(
            "Connectivity types (top-2 blended)",
            "Orientation (dominant neighbour dipole orientation θ)",
            "Crystallinity: q<sub>6</sub>-ordered fraction",
            "Crystallinity: q<sub>4</sub>-ordered fraction",
        ),
    )

    fig.add_trace(
        topology_heatmap,
        row=1,
        col=1,
    )

    fig.add_trace(
        orientation_heatmap,
        row=1,
        col=2,
    )

    fig.add_trace(
        q6_heatmap,
        row=1,
        col=3,
    )

    fig.add_trace(
        q4_heatmap,
        row=2,
        col=3,
    )

    for trace in topology_legend:
        fig.add_trace(
            trace,
            row=1,
            col=1,
        )

    # Full-height panels
    fig.update_xaxes(
        title_text="shift",
        tickfont=dict(size=TICK_FONT),
        title_font=dict(size=TITLE_FONT),
        row=1,
        col=1,
    )

    fig.update_xaxes(
        title_text="shift",
        tickfont=dict(size=TICK_FONT),
        title_font=dict(size=TITLE_FONT),
        row=1,
        col=2,
    )

    fig.update_yaxes(
        title_text="λ",
        tickfont=dict(size=TICK_FONT),
        title_font=dict(size=TITLE_FONT),
        row=1,
        col=1,
    )

    fig.update_yaxes(
        title_text="λ",
        tickfont=dict(size=TICK_FONT),
        title_font=dict(size=TITLE_FONT),
        row=1,
        col=2,
    )

    # q6 panel
    fig.update_xaxes(
        title_text="",
        showticklabels=True,
        tickfont=dict(size=TICK_FONT),
        row=1,
        col=3,
    )

    fig.update_yaxes(
        title_text="λ",
        tickfont=dict(size=TICK_FONT),
        title_font=dict(size=TITLE_FONT),
        row=1,
        col=3,
    )

    # q4 panel
    fig.update_xaxes(
        title_text="shift",
        tickfont=dict(size=TICK_FONT),
        title_font=dict(size=TITLE_FONT),
        row=2,
        col=3,
    )

    fig.update_yaxes(
        title_text="λ",
        tickfont=dict(size=TICK_FONT),
        title_font=dict(size=TITLE_FONT),
        row=2,
        col=3,
    )

    fig.update_annotations(
        font=dict(size=TITLE_FONT)
    )

    # Position the two visible colorbars.
    x2_right = fig.layout.xaxis2.domain[1]
    x3_right = fig.layout.xaxis3.domain[1]

    fig.update_traces(
        selector=dict(
            type="heatmap",
            name="orientation",
        ),
        colorbar=dict(
            x=x2_right,
            xanchor="left",
            y=0.5,
            yanchor="middle",
            len=0.90,
            thickness=18,
            tickvals=[
                0,
                np.pi / 4,
                np.pi / 2,
                3 * np.pi / 4,
                np.pi,
            ],
            ticktext=[
                "0",
                "π/4",
                "π/2",
                "3π/4",
                "π",
            ],
        ),
    )

    # Shared q6/q4 interpretation:
    # only q6 displays the bar, but q4 uses the identical scale.
    fig.update_traces(
        selector=dict(
            type="heatmap",
            name="q6 crystallinity",
        ),
        colorbar=dict(
            title=dict(
                text="ordered fraction",
                side="right",
            ),
            x=x3_right,
            xanchor="left",
            y=0.5,
            yanchor="middle",
            len=0.90,
            thickness=18,
            tickvals=np.linspace(
                cryst_vmin,
                cryst_vmax,
                5,
            ),
        ),
    )

    fig.update_layout(
        template="plotly_white",
        width=width,
        height=height,
        margin=dict(
            l=70,
            r=70,
            t=50,
            b=50,
        ),
        legend=dict(
            title="Connectivity types",
            yanchor="top",
            y=0.95,
            xanchor="left",
            x=0.005,
            font=dict(size=13),
        ),
    )

    return fig


# ============================================================
# Create, show and export
# ============================================================
fig = make_combined_state_diagram(
    df1,
    df2,
    df3,
    df4,
    orientation_cmap="cmr.swamp",
    cryst_cmap="cmr.cosmic",
    cryst_vmin=0.0,
    cryst_vmax=0.4,
    width=1650,
    height=480,
)

fig.show()

fig.write_image(
    "Figure-3-combined_state_diagram.pdf",
    width=1650,
    height=480,
    scale=1,
)